# Hisse Fiyati - LSTM

AAPL kapanisi. Day5 Sequential, zaman icin LSTM.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


### Data


In [ ]:
df=pd.read_csv('data/AAPL.csv',parse_dates=['Date']).sort_values('Date')
df.head()


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


### Bos veri


In [ ]:
df['Close']=df['Close'].ffill()


### Gorsellestirme


In [ ]:
plt.plot(df['Date'],df['Close'])
plt.title('AAPL Close')
plt.show()


### Sequence (dongu yok)


In [ ]:
from sklearn.preprocessing import MinMaxScaler
sc=MinMaxScaler()
c=sc.fit_transform(df[['Close']]).ravel()
win=20
X=np.lib.stride_tricks.sliding_window_view(c[:-1],win)
y=c[win:]
X=X.reshape(-1,win,1)
split=int(len(X)*0.8)
x_train,x_test=X[:split],X[split:]
y_train,y_test=y[:split],y[split:]
x_train.shape


### 1. LSTM


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU


In [ ]:
m1=Sequential()
m1.add(LSTM(32,input_shape=(win,1)))
m1.add(Dense(1))
m1.compile(loss='mse',optimizer='adam')
m1.fit(x_train,y_train,validation_data=(x_test,y_test),epochs=8,batch_size=32,verbose=1)
print('LSTM mse',m1.evaluate(x_test,y_test,verbose=0))


### 2. GRU


In [ ]:
m2=Sequential()
m2.add(GRU(32,input_shape=(win,1)))
m2.add(Dense(1))
m2.compile(loss='mse',optimizer='adam')
m2.fit(x_train,y_train,validation_data=(x_test,y_test),epochs=8,batch_size=32,verbose=1)
print('GRU mse',m2.evaluate(x_test,y_test,verbose=0))


### 3. Dense (Day5 ev fiyati tarzi)


In [ ]:
m3=Sequential()
m3.add(Dense(64,activation='relu'))
m3.add(Dense(32,activation='relu'))
m3.add(Dense(1))
m3.compile(loss='mse',optimizer='adam')
m3.fit(x_train.reshape(-1,win),y_train,validation_data=(x_test.reshape(-1,win),y_test),epochs=8,batch_size=32,verbose=1)
print('Dense mse',m3.evaluate(x_test.reshape(-1,win),y_test,verbose=0))


In [ ]:
pred=m1.predict(x_test,verbose=0)
plt.plot(y_test,label='gercek')
plt.plot(pred.ravel(),label='LSTM')
plt.legend()
plt.show()


In [ ]:
m1.save('../../models/dl_stock_lstm.h5')


### Sonuc

LSTM/GRU seri icin uygun. Son gunleri fena takip ediyor. Hedefi tutturdum.
